<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P03/pratica03_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prática 3 — Anotação Genômica

**Mapeamento de transcritos no genoma e predição gênica *ab initio* (AUGUSTUS & ANNEVO)**

Biologia Molecular Computacional — 2026/2

---

## Objetivos de aprendizagem
Ao final desta prática você deverá ser capaz de:
1. Identificar o organismo e o *locus* de origem de um transcrito desconhecido por alinhamento ao genoma.
2. Deduzir a estrutura éxon/íntron de um gene a partir de um alinhamento *spliced*.
3. Executar predição gênica *ab initio* (AUGUSTUS e ANNEVO) e interpretar arquivos GFF/GFF3.
4. Comparar a predição feita sobre o **transcrito maduro** vs. sobre o **DNA genômico** e explicar a diferença.
5. Confrontar as predições com a anotação de referência (RefSeq).

## Pré-requisitos conceituais
mRNA maduro e *splicing* · íntron/éxon · CDS × UTR · sinal de poliadenilação · predição *ab initio* × baseada em evidência · formatos FASTA / GFF3.

## Tempo estimado
~2 h (a etapa da ANNEVO exige GPU).

## Entregáveis
Responda as questões **Q1–Q12** ao longo do notebook e entregue o relatório final (modelo na última célula) em PDF.

> ### ⚠️ Antes de começar
> Ative a GPU: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware → GPU (T4)**.
> Depois rode as células em ordem.


## 0. Configuração do ambiente

As células abaixo instalam tudo o que o notebook usa:

| Ferramenta | Uso nesta prática |
|---|---|
| **Biopython** | baixar sequências do NCBI (Entrez), rodar BLAST online opcional |
| **NCBI BLAST+** | `blastp` para o passo “Blast2seq” entre proteínas preditas |
| **AUGUSTUS** | predição gênica *ab initio* clássica (modelo `human`) |
| **ANNEVO** | predição gênica *ab initio* por *deep learning* (modelo `Mammalia`) |
| **gffutils** | leitura de GFF3 e tradução de CDS |


In [1]:
# 0.1 — Conferência da GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU visível:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Ative a GPU em Ambiente de execução -> Alterar o tipo de ambiente de execução."


Tesla T4, 15360 MiB, 580.82.07
torch: 2.11.0+cu128 | CUDA: 12.8 | GPU visível: True


In [2]:
# 0.2 — Instalação (leva ~3-5 min)
%%bash
set -e
apt-get -qq update
apt-get -qq install -y augustus augustus-data ncbi-blast+ > /dev/null
pip -q install biopython gffutils "bcbio-gff==0.7.1" "h5py>=3.1" numba tqdm
pip -q install "torchmetrics==0.8.2" 2>/dev/null || pip -q install torchmetrics
echo "AUGUSTUS : $(augustus --version 2>&1 | head -n1)"
echo "blastp   : $(blastp -version | head -n1)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 29.7 MB/s eta 0:00:00
AUGUSTUS : AUGUSTUS (3.4.0) is a gene prediction tool.
blastp   : blastp: 2.12.0+


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
# 0.3 — Parâmetros globais
import os
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

# O NCBI EXIGE um e-mail válido para uso programático do Entrez:
Entrez.email = "SEU_EMAIL@usp.br"          # <-- EDITE

os.environ["AUGUSTUS_CONFIG_PATH"] = "/usr/share/augustus/config"
WORK = "/content/pratica3"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("Diretório de trabalho:", os.getcwd())


Diretório de trabalho: /content/pratica3


In [4]:
# 0.4 — Funções auxiliares (execute e siga em frente)
from collections import Counter
import gffutils, subprocess
from Bio import SeqIO
from Bio.Seq import Seq

def summarize_gff(path, label=None):
    # Conta features por tipo e mostra o intervalo de cada gene/mRNA.
    types, spans = Counter(), []
    with open(path) as fh:
        for ln in fh:
            if ln.startswith("#") or not ln.strip():
                continue
            f = ln.rstrip("\n").split("\t")
            if len(f) < 8:
                continue
            types[f[2]] += 1
            if f[2] in ("gene", "mRNA", "transcript"):
                spans.append((f[2], int(f[3]), int(f[4]), f[6]))
    print(f"== {label or path} ==")
    for t, c in sorted(types.items()):
        print(f"   {t:16s} {c}")
    for t, a, b, st in spans:
        print(f"   -> {t} {a}-{b} ({st})  {b - a + 1} pb")
    return types

def augustus_proteins(path):
    # Extrai as sequencias de proteina dos comentarios do AUGUSTUS.
    seqs, cur, on = [], [], False
    for ln in open(path):
        if not ln.startswith("#"):
            continue
        s = ln.strip("#").strip()
        if s.startswith("protein sequence = ["):
            on, s = True, s.split("[", 1)[1]
        if on:
            cur.append(s.replace("]", "").strip())
            if "]" in ln:
                seqs.append("".join(cur).replace(" ", ""))
                cur, on = [], False
    return [p for p in seqs if p]

def proteins_from_gff(gff_path, genome_fa):
    # Traduz o(s) CDS de um GFF/GFF3 usando o genomico correspondente.
    genome = {r.id: r.seq for r in SeqIO.parse(genome_fa, "fasta")}
    db = gffutils.create_db(gff_path, ":memory:", force=True, keep_order=True,
                            merge_strategy="create_unique")
    prots = []
    for m in db.features_of_type(("mRNA", "transcript")):
        cds = list(db.children(m, featuretype="CDS", order_by="start"))
        if not cds:
            continue
        seq = "".join(str(genome[c.seqid][c.start - 1:c.end]) for c in cds)
        if cds[0].strand == "-":
            seq = str(Seq(seq).reverse_complement())
        prots.append(str(Seq(seq).translate(to_stop=True)))
    return prots

def _write_fa(seqs, path, prefix):
    with open(path, "w") as fh:
        for i, s in enumerate(seqs, 1):
            fh.write(f">{prefix}{i}\n{s}\n")
    return path

def blast2seq_prot(seqs_a, seqs_b, label_a="A", label_b="B"):
    # Equivalente ao 'Align two or more sequences' do BLAST, para proteinas.
    if not seqs_a or not seqs_b:
        print("(uma das listas de proteínas está vazia)"); return
    a = _write_fa(seqs_a, "protA.fa", label_a)
    b = _write_fa(seqs_b, "protB.fa", label_b)
    out = subprocess.run(
        ["blastp", "-query", a, "-subject", b, "-outfmt",
         "6 qseqid sseqid pident length mismatch gapopen evalue bitscore"],
        capture_output=True, text=True).stdout
    print("qseq  sseq  %id   len  mism gap   evalue     bits")
    print(out.strip() or "(sem alinhamento significativo)")


## 1. A sequência-problema

Você recebeu a sequência de um transcrito (mRNA) e precisa descobrir:

- a que **organismo** ela pertence;
- de que **região do genoma** foi transcrita;
- se o gene completo contém **íntrons** e, em caso afirmativo, quais são as regiões de **íntrons/éxons**.


    "GCCGGAGCGGATCGCGGAGTTTCACCGCGGACCTGTCAGAGATACAGAGGTTGTGGGGGCGGGAGACAGAAAGAGAGAGAGATCCAGAGACCGAGTCTTACGTTGACACGCAGAGAGAAAGACGCAGAGACAGACAAACAAACAGATAGGAGAGGCTCTCCAGGAGGCCGGGGGGCCCACTCCGCCTATCGCTCCCCTCGGCTACGCTGCCACTTCAATGCCCCGCAGGTCGCGAGCTGCTGTTCTTTCGAAGGCGTCGGAGAACCAGGGGCGTCCCGCGCCACCTCTGACTCGGAGCAGCGCCGAGCACTGACGCTCCCGCCCTTGGGCAAGGACGCCAGTGCGCCCGCGCGCGTCCCTCTGCGCGGCAGCCCGTCGCGGGCCCTCAAGGGGAAGCCCAGGCCAGGATGGCCCCGGGTCGCGCGGTGGCCGGGCTCCTGTTGCTGGCGGCCGCCGGCCTCGGAGGAGTGGCGGAGGGGCCAGGGCTAGCCTTCAGCGAGGATGTGCTGAGCGTGTTCGGCGCGAATCTGAGCCTGTCGGCGGCGCAGCTCCAGCACTTGCTGGAGCAGATGGGAGCCGCCTCCCGCGTGGGCGTCCCGGAGCCTGGCCAGCTGCACTTCAACCAGTGTTTAACTGCTGAAGAGATCTTTTCCCTTCATGGCTTTTCAAATGCTACCCAAATAACCAGCTCCAAATTCTCTGTCATCTGTCCAGCAGTCTTACAGCAATTGAACTTTCACCCATGTGAGGATCGGCCCAAGCACAAAACAAGACCAAGTCATTCAGAAGTTTGGGGATATGGATTCCTGTCAGTGACGATTATTAATCTGGCATCTCTCCTCGGATTGATTTTGACTCCACTGATAAAGAAATCTTATTTCCCAAAGATTTTGACCTTTTTTGTGGGGCTGGCTATTGGGACTCTTTTTTCAAATGCAATTTTCCAACTTATTCCAGAGGCATTTGGATTTGATCCCAAAGTCGACAGTTATGTTGAGAAGGCAGTTGCTGTGTTTGGTGGATTTTACCTACTTTTCTTTTTTGAAAGAATGCTAAAGATGTTATTAAAGACATATGGTCAGAATGGTCATACCCACTTTGGAAATGATAACTTTGGTCCTCAAGAAAAAACTCATCAACCTAAAGCATTACCTGCCATCAATGGTGTGACATGCTATGCAAATCCTGCTGTCACAGAAGCTAATGGACATATCCATTTTGATAATGTCAGTGTGGTATCTCTACAGGATGGAAAAAAAGAGCCAAGTTCATGTACCTGTTTGAAGGGGCCCAAACTGTCAGAAATAGGGACGATTGCCTGGATGATAACGCTCTGCGATGCCCTCCACAATTTCATCGATGGCCTGGCGATTGGGGCTTCCTGCACCTTGTCTCTCCTTCAGGGACTCAGTACTTCCATAGCAATCCTATGTGAGGAGTTTCCCCACGAGTTAGGAGACTTTGTGATCCTACTCAATGCAGGGATGAGCACTCGACAAGCCTTGCTATTCAACTTCCTTTCTGCATGTTCCTGCTATGTTGGGCTAGCTTTTGGCATTTTGGTGGGCAACAATTTCGCTCCAAATATTATATTTGCACTTGCTGGAGGCATGTTCCTCTATATTTCTCTGGCAGATATGTTTCCAGAGATGAATGATATGCTGAGAGAAAAGGTAACTGGAAGAAAAACCGATTTCACCTTCTTCATGATTCAGAATGCTGGAATGTTAACTGGATTCACAGCCATTCTACTCATTACCTTGTATGCAGGAGAAATCGAATTGGAGTAATAGAAAATGGAAGATGGTGTTGTTAATAAAGGCATTTAATAGATAAAAACATCTCCAAAAAGGATTTTGAAGCTGATCCTATTTAGTTAAAAAGATAATTTTGCTTTCAACTGTAGGTCCAGAAAACTAATTATTGGCATCAGTCTGTGAAATAGTCCATTATTTGTTGTTAAAAATGCTTCAAAAGGTTTTCAGTGTCAGTCTGAGATGCCTGGTATATAGGAGCCTTTGGGAAATACCTATTTTTCAGTATTCCATGCATATTAGATATCACCATGAAGCAAGAGACATGCATTCTATAATCATGTAGACACTCAGACTCAGGGGAAAATACAAGTTATATCCTGAAAGCCTTTAAAACTCTATGGTAGGATCAAAGATTCAAATGGTTTCAGAGAGGTTTTATTTCAATTAATTTGTTCTAGTGCTTTCAAGAGCAAGTACATCAAAATGTAGAAGGTAAAATGTATGCAACACTAATATAAATTATTCCAAGTCTTTAAGGAGCCAAAGAAAAAAAAGATTTCTCACAGCTTTTTGTTCTGTTTTGTATTTCAATTAGGAACTTGCAGTATTATTTTGAAAACCATTCTAAAATAATAGGAGTTAGGAAATAAATAAAGTTTTGCTAGCCCTGCTAAGTTCAGGCTTAGAGGCTTATCGCTAAGTATAAACTTCACCAGATTCCACGAAAAGCTGGATAGCTTTTTTTCTGACTTATGTTGTGGTTGCACCCCTCACAAATGGCAGAACAGTATGTAAAGCTGGTAACACCTCGGTTTCAGTGCACCATGTGTTTGCTTTGTGAAGGTGAAGAATATGTTGGTTTAGAGAAAGAAATTGGATGTAATTTTATGCAATTTACTTTTAAAGACAAACATAACTATTTAGCAGAGAATATTTTAATAAATGCAAAACAACAGCTGGACTGCTGTACATCAAGGACAGATTAACTGGAAAACATATGTTCCTTATGTGTGATCGAGAGCCATTCAGAAAAGACTTCCTTTGTGTTCAGCCTATACTTTTCCATATGGTATACCTTGAAAAAAATTAGCACACCATGGTTATTTTTCTACCTTTTATAAAAGACAGAGCCTGTTTACTCATTTAGAAGATAGAGAAAATTGGTCTAAAATTGAACATCCTAGATTCACACTCCCAAGTCACTTAAGGTGATTTGATGGTGAGGAAAATGATTGACAAAGCCCAACAATGATCTCAGGAATTACATTTTCCAACAGACCAAAAAATGTTTTCATGTAGCAGCAATGCAGATTTGGTGAATATTTAATATATATTTTAGTATGTATTTCACTTTATGACTGACAATTAAAAAATATTGTTTGGCCAAATAGTAAACACCCTTTTGAAACCATGAAAAAAAAAAAAAAAAA"


In [5]:
# 1.1 — Grava o transcrito em FASTA
mrna = (
    "GCCGGAGCGGATCGCGGAGTTTCACCGCGGACCTGTCAGAGATACAGAGGTTGTGGGGGCGGGAGACAGA"
    "AAGAGAGAGAGATCCAGAGACCGAGTCTTACGTTGACACGCAGAGAGAAAGACGCAGAGACAGACAAACA"
    "AACAGATAGGAGAGGCTCTCCAGGAGGCCGGGGGGCCCACTCCGCCTATCGCTCCCCTCGGCTACGCTGC"
    "CACTTCAATGCCCCGCAGGTCGCGAGCTGCTGTTCTTTCGAAGGCGTCGGAGAACCAGGGGCGTCCCGCG"
    "CCACCTCTGACTCGGAGCAGCGCCGAGCACTGACGCTCCCGCCCTTGGGCAAGGACGCCAGTGCGCCCGC"
    "GCGCGTCCCTCTGCGCGGCAGCCCGTCGCGGGCCCTCAAGGGGAAGCCCAGGCCAGGATGGCCCCGGGTC"
    "GCGCGGTGGCCGGGCTCCTGTTGCTGGCGGCCGCCGGCCTCGGAGGAGTGGCGGAGGGGCCAGGGCTAGC"
    "CTTCAGCGAGGATGTGCTGAGCGTGTTCGGCGCGAATCTGAGCCTGTCGGCGGCGCAGCTCCAGCACTTG"
    "CTGGAGCAGATGGGAGCCGCCTCCCGCGTGGGCGTCCCGGAGCCTGGCCAGCTGCACTTCAACCAGTGTT"
    "TAACTGCTGAAGAGATCTTTTCCCTTCATGGCTTTTCAAATGCTACCCAAATAACCAGCTCCAAATTCTC"
    "TGTCATCTGTCCAGCAGTCTTACAGCAATTGAACTTTCACCCATGTGAGGATCGGCCCAAGCACAAAACA"
    "AGACCAAGTCATTCAGAAGTTTGGGGATATGGATTCCTGTCAGTGACGATTATTAATCTGGCATCTCTCC"
    "TCGGATTGATTTTGACTCCACTGATAAAGAAATCTTATTTCCCAAAGATTTTGACCTTTTTTGTGGGGCT"
    "GGCTATTGGGACTCTTTTTTCAAATGCAATTTTCCAACTTATTCCAGAGGCATTTGGATTTGATCCCAAA"
    "GTCGACAGTTATGTTGAGAAGGCAGTTGCTGTGTTTGGTGGATTTTACCTACTTTTCTTTTTTGAAAGAA"
    "TGCTAAAGATGTTATTAAAGACATATGGTCAGAATGGTCATACCCACTTTGGAAATGATAACTTTGGTCC"
    "TCAAGAAAAAACTCATCAACCTAAAGCATTACCTGCCATCAATGGTGTGACATGCTATGCAAATCCTGCT"
    "GTCACAGAAGCTAATGGACATATCCATTTTGATAATGTCAGTGTGGTATCTCTACAGGATGGAAAAAAAG"
    "AGCCAAGTTCATGTACCTGTTTGAAGGGGCCCAAACTGTCAGAAATAGGGACGATTGCCTGGATGATAAC"
    "GCTCTGCGATGCCCTCCACAATTTCATCGATGGCCTGGCGATTGGGGCTTCCTGCACCTTGTCTCTCCTT"
    "CAGGGACTCAGTACTTCCATAGCAATCCTATGTGAGGAGTTTCCCCACGAGTTAGGAGACTTTGTGATCC"
    "TACTCAATGCAGGGATGAGCACTCGACAAGCCTTGCTATTCAACTTCCTTTCTGCATGTTCCTGCTATGT"
    "TGGGCTAGCTTTTGGCATTTTGGTGGGCAACAATTTCGCTCCAAATATTATATTTGCACTTGCTGGAGGC"
    "ATGTTCCTCTATATTTCTCTGGCAGATATGTTTCCAGAGATGAATGATATGCTGAGAGAAAAGGTAACTG"
    "GAAGAAAAACCGATTTCACCTTCTTCATGATTCAGAATGCTGGAATGTTAACTGGATTCACAGCCATTCT"
    "ACTCATTACCTTGTATGCAGGAGAAATCGAATTGGAGTAATAGAAAATGGAAGATGGTGTTGTTAATAAA"
    "GGCATTTAATAGATAAAAACATCTCCAAAAAGGATTTTGAAGCTGATCCTATTTAGTTAAAAAGATAATT"
    "TTGCTTTCAACTGTAGGTCCAGAAAACTAATTATTGGCATCAGTCTGTGAAATAGTCCATTATTTGTTGT"
    "TAAAAATGCTTCAAAAGGTTTTCAGTGTCAGTCTGAGATGCCTGGTATATAGGAGCCTTTGGGAAATACC"
    "TATTTTTCAGTATTCCATGCATATTAGATATCACCATGAAGCAAGAGACATGCATTCTATAATCATGTAG"
    "ACACTCAGACTCAGGGGAAAATACAAGTTATATCCTGAAAGCCTTTAAAACTCTATGGTAGGATCAAAGA"
    "TTCAAATGGTTTCAGAGAGGTTTTATTTCAATTAATTTGTTCTAGTGCTTTCAAGAGCAAGTACATCAAA"
    "ATGTAGAAGGTAAAATGTATGCAACACTAATATAAATTATTCCAAGTCTTTAAGGAGCCAAAGAAAAAAA"
    "AGATTTCTCACAGCTTTTTGTTCTGTTTTGTATTTCAATTAGGAACTTGCAGTATTATTTTGAAAACCAT"
    "TCTAAAATAATAGGAGTTAGGAAATAAATAAAGTTTTGCTAGCCCTGCTAAGTTCAGGCTTAGAGGCTTA"
    "TCGCTAAGTATAAACTTCACCAGATTCCACGAAAAGCTGGATAGCTTTTTTTCTGACTTATGTTGTGGTT"
    "GCACCCCTCACAAATGGCAGAACAGTATGTAAAGCTGGTAACACCTCGGTTTCAGTGCACCATGTGTTTG"
    "CTTTGTGAAGGTGAAGAATATGTTGGTTTAGAGAAAGAAATTGGATGTAATTTTATGCAATTTACTTTTA"
    "AAGACAAACATAACTATTTAGCAGAGAATATTTTAATAAATGCAAAACAACAGCTGGACTGCTGTACATC"
    "AAGGACAGATTAACTGGAAAACATATGTTCCTTATGTGTGATCGAGAGCCATTCAGAAAAGACTTCCTTT"
    "GTGTTCAGCCTATACTTTTCCATATGGTATACCTTGAAAAAAATTAGCACACCATGGTTATTTTTCTACC"
    "TTTTATAAAAGACAGAGCCTGTTTACTCATTTAGAAGATAGAGAAAATTGGTCTAAAATTGAACATCCTA"
    "GATTCACACTCCCAAGTCACTTAAGGTGATTTGATGGTGAGGAAAATGATTGACAAAGCCCAACAATGAT"
    "CTCAGGAATTACATTTTCCAACAGACCAAAAAATGTTTTCATGTAGCAGCAATGCAGATTTGGTGAATAT"
    "TTAATATATATTTTAGTATGTATTTCACTTTATGACTGACAATTAAAAAATATTGTTTGGCCAAATAGTA"
    "AACACCCTTTTGAAACCATGAAAAAAAAAAAAAAAAA"
)

with open("transcrito.fa", "w") as fh:
    fh.write(">transcrito_problema\n")
    for i in range(0, len(mrna), 70):
        fh.write(mrna[i:i + 70] + "\n")

n_polyA = len(mrna) - len(mrna.rstrip("A"))
print("Tamanho do transcrito :", len(mrna), "nt")
print("Cauda poli-A no 3'    :", n_polyA, "A consecutivos")
print(open("transcrito.fa").read()[:220], "...")


Tamanho do transcrito : 3187 nt
Cauda poli-A no 3'    : 17 A consecutivos
>transcrito_problema
GCCGGAGCGGATCGCGGAGTTTCACCGCGGACCTGTCAGAGATACAGAGGTTGTGGGGGCGGGAGACAGA
AAGAGAGAGAGATCCAGAGACCGAGTCTTACGTTGACACGCAGAGAGAAAGACGCAGAGACAGACAAACA
AACAGATAGGAGAGGCTCTCCAGGAGGCCGGGGGGCCCACTCCGCCTATCGCTCCCC ...


In [6]:
# 1.2 — Procura o sinal clássico de poliadenilação (AAUAAA) perto do 3'
import re
for m in re.finditer("AATAAA", mrna):
    print(f"AATAAA na posição {m.start() + 1:5d}  (a {len(mrna) - m.start()} nt do fim)")
# Q5: qual desses é o sinal de poli-A funcional? Qual o tamanho aproximado da 3'UTR?


AATAAA na posição  1815  (a 1373 nt do fim)
AATAAA na posição  2403  (a 785 nt do fim)
AATAAA na posição  2695  (a 493 nt do fim)


## 2. Identificação por BLASTN (contra o genoma humano)

Faça o **Caminho A no navegador.** Ali vamos entender a interface e *ver* a estrutura do gene.

---

### Caminho A — passo a passo no site do NCBI

1. Abra o **[NCBI BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi)** e clique em **blastn**
   (*Nucleotide BLAST*).
2. No campo **Enter Query Sequence**, cole a sequência do transcrito (o conteúdo de
   `transcrito.fa`, gerado na célula 1.1 — ou a sequência do enunciado).
3. Em **Choose Search Set → Database**, selecione **“RefSeq Representative genomes”**.
4. No campo **Organism**, digite **`Homo sapiens`** e selecione a sugestão (taxid:9606).
5. Abra **Algorithm parameters** (rodapé do formulário) e, em **Filters and Masking →
   Filter**, **desmarque** *“Low complexity regions”*.
   > Os ajustes 3, 4 e 5 restringem a busca às versões depositadas do genoma humano e
   > impedem que trechos de baixa complexidade sejam mascarados, permitindo o alinhamento
   > completo do transcrito.
6. Clique no botão azul **BLAST** e aguarde.

### O que observar na página de resultados

7. **Aba *Descriptions*** — o melhor *hit* deve ser um cromossomo humano
   (`NC_0000NN.NN Homo sapiens chromosome NN, GRCh38.p14`). Anote o **número de acesso**
   e o nome do **gene** associado (coluna *Description* / link *Gene*). **(Q4)**
8. **Aba *Graphic Summary*** — o alinhamento aparece como **vários blocos vermelhos
   separados por linhas finas**: cada bloco é um **éxon** e cada linha fina é um **íntron**.
   Conte os blocos. **(Q2)**
9. **Aba *Alignments*** — para o *hit* do cromossomo:
   - **`Range 1: <início>..<fim>`** → são as coordenadas do gene no cromossomo. O **menor**
     valor é o início e o **maior** é o fim. **(Q1)**
   - **`Strand=Plus/Plus`** ou **`Strand=Plus/Minus`** → diz em que **fita** o gene está.
     Plus/Minus significa que o gene é transcrito da fita reversa. **(Q3)**
   - Role pelo alinhamento: onde a numeração do *Sbjct* **pula centenas/milhares de bases**
     de uma linha para a outra, há um **íntron**; o texto contínuo é **éxon**.

### Obter a sequência genômica pelo site

10. Ainda em *Alignments*, clique no **hiperlink do número de acesso** do *hit* — abre o
    registro do **cromossomo inteiro** no *Nucleotide*.
11. No canto superior direito, em **Change region shown**, marque **Selected region** e
    digite os valores de **início** e **fim** que você anotou na Q1 (pode somar 40.000 pb de
    folga em cada ponta).
12. Em **Customize view / Display options**, ou no menu **Send to → File**, escolha o
    formato **FASTA**.
13. A sequência FASTA que aparece é a **região do genoma que contém o seu gene**. É
    exatamente essa região que a célula 3.1 baixa automaticamente pelo Entrez (com flanco),
    para ser usada no AUGUSTUS Parte 2 e na ANNEVO.

> **Observação importante:** o blastn **não é *splice-aware***. Os limites dos HSPs são
> *aproximações* dos éxons — os sítios de *splice* exatos (GT…AG) e éxons muito curtos
> podem não aparecer. Para uma estrutura mais precisa, alinhe o transcrito ao genoma com
> **[BLAT (UCSC)](https://genome.ucsc.edu/cgi-bin/hgBlat)** ou **NCBI Splign**.

---

  
  
  
  
### Caminho B — mesma busca, por código (opcional, mais lento)
O **Caminho B** (célula 2.1) automatiza a mesma busca;
- a célula 3.1 baixa a região genômica para você, dispensando o recorte manual
- Os passos 8–11 acima ajudam a para entender **de onde** vêm as coordenadas.



In [7]:
# 2.1 — BLASTN online opcional (~2-5 min). Deixe False se já fez no site.
RUN_ONLINE_BLAST = False

if RUN_ONLINE_BLAST:
    from Bio.Blast import NCBIWWW, NCBIXML
    handle = NCBIWWW.qblast(
        "blastn", "refseq_representative_genomes", mrna,
        entrez_query="Homo sapiens[Organism]", hitlist_size=5,
        megablast=True, expect=1e-20)
    rec = NCBIXML.read(handle)
    for al in rec.alignments[:3]:
        print(al.accession, "|", al.hit_def[:70])
        for h in sorted(al.hsps, key=lambda x: x.sbjct_start):
            strand = "Plus/Plus" if h.frame[1] > 0 else "Plus/Minus"
            print(f"   query {h.query_start:>5}-{h.query_end:<5}  "
                  f"subject {h.sbjct_start:>10}-{h.sbjct_end:<10}  "
                  f"id {h.identities}/{h.align_length}  {strand}")


### Q1–Q4 — anote os resultados do BLAST

- **Q1.** Qual o cromossomo (número de acesso) e em que base o gene **começa** e **termina**?
- **Q2.** Quantos **éxons** o alinhamento sugere (número de HSPs)?
- **Q3.** O alinhamento é **Plus/Plus** ou **Plus/Minus**? Em que **fita** do cromossomo está o gene?
- **Q4.** Qual é o **gene** e sua **função**? (use a definição do *hit*; confirme depois com um blastp da proteína predita)

Preencha as variáveis abaixo com os valores do **seu** BLAST:


In [8]:
# 2.2 — EDITE com o resultado do seu BLAST
#
# Os valores abaixo sao os de REFERENCIA (assembly GRCh38.p14) para conferencia.
# O aluno deve chegar a numeros parecidos lendo o bloco "Alignments" do proprio BLAST
# (linha "Range 1: <inicio>..<fim>" do hit do cromossomo). Pequenas diferencas de algumas
# centenas de pb nas pontas (5'/3' UTR) sao normais.
#
# Transcrito = NM_001135146.2  ->  gene SLC39A8 (ZIP8), transportador de zinco/manganes,
# cromossomo 4q24, fita MINUS.

ACCESSION  = "NC_000004.12"   # cromossomo 4, GRCh38.p14
GENE_START = 102251041        # extremidade 3' do gene no cromossomo  (Q1)
GENE_END   = 102345482        # extremidade 5' do gene no cromossomo  (Q1)
STRAND     = "-"              # Plus/Minus no BLAST -> gene na fita reversa  (Q3)
FLANK      = 40000            # pb de folga em CADA extremidade.
                             # AUGUSTUS funciona com pouco (~1000); a ANNEVO precisa de
                             # contexto grande (janela do modelo ~102 kb) ou a saida vem
                             # VAZIA. Aqui o proprio gene ja tem ~94 kb, entao 40 kb bastam.

assert GENE_START and GENE_END and GENE_END > GENE_START, \
    "Preencha GENE_START/GENE_END com as coordenadas do seu BLAST."
print(f"Regiao alvo: {ACCESSION}:{GENE_START}-{GENE_END} ({STRAND}), +/- {FLANK} pb de flanco")
print(f"Gene: ~{(GENE_END - GENE_START)/1000:.0f} kb | regiao a baixar: ~{(GENE_END - GENE_START + 2*FLANK)/1000:.0f} kb")


Regiao alvo: NC_000004.12:102251041-102345482 (-), +/- 40000 pb de flanco
Gene: ~94 kb | regiao a baixar: ~174 kb


## 3. Obtenção da região genômica

Em vez de recortar o cromossomo à mão no site, baixamos a região diretamente pelo Entrez, **com flanco**.

**Q: Qual a importância desse flanco?**


In [9]:
# 3.1 — Baixa a região genômica (FASTA) e a mesma região com anotações (GenBank)
start = max(1, GENE_START - FLANK)
stop  = GENE_END + FLANK

h = Entrez.efetch(db="nuccore", id=ACCESSION, rettype="fasta", retmode="text",
                  seq_start=start, seq_stop=stop, strand=1)
rec = SeqIO.read(h, "fasta")
rec.id, rec.description = f"{ACCESSION}:{start}-{stop}", ""
SeqIO.write(rec, "genomico.fa", "fasta")
print("Região genômica :", len(rec.seq), "pb  ->", rec.id)

h = Entrez.efetch(db="nuccore", id=ACCESSION, rettype="gbwithparts", retmode="text",
                  seq_start=start, seq_stop=stop, strand=1)
open("genomico.gb", "w").write(h.read())
print("GenBank com features salvo em genomico.gb")
# Mantemos a fita '+' do cromossomo para as coordenadas baterem com o RefSeq;
# AUGUSTUS e ANNEVO preveem os dois sentidos de qualquer forma.


Região genômica : 174442 pb  -> NC_000004.12:102211041-102385482
GenBank com features salvo em genomico.gb


## 4. AUGUSTUS — Parte 1: predição sobre o **transcrito**

Rodamos o AUGUSTUS (modelo `human`) diretamente sobre o mRNA maduro.

> **Alternativa visual:** o mesmo pode ser feito no
> [servidor web do AUGUSTUS](https://bioinf.uni-greifswald.de/augustus/submission.php) —
> cole a sequência em *“Paste your sequence(s) here”*, escolha **Organism → *Homo sapiens***
> e clique em *Run AUGUSTUS*. A célula abaixo só automatiza isso e já conta os éxons.


In [10]:
# 4.1
!augustus --species=human --gff3=on --UTR=off transcrito.fa > aug_transcrito.gff3 2> aug_transcrito.log
tipos_tx = summarize_gff("aug_transcrito.gff3", "AUGUSTUS - transcrito (Parte 1)")
prot_tx = augustus_proteins("aug_transcrito.gff3")
print("\nProteína(s) predita(s):", [len(p) for p in prot_tx], "aa")
# Q6: quantos e quais tipos de exons? Há introns? Por que?
# Q7: tamanho total dos exons (= tamanho do CDS + UTR previsto).


== AUGUSTUS - transcrito (Parte 1) ==
   CDS              1
   gene             1
   start_codon      1
   stop_codon       1
   transcript       1
   -> gene 408-1790 (+)  1383 pb
   -> transcript 408-1790 (+)  1383 pb

Proteína(s) predita(s): [460] aa


## 5. AUGUSTUS — Parte 2: predição sobre o **genômico**

Mesma ferramenta, agora sobre a região genômica baixada na célula 3.1 (com íntrons).


In [11]:
# 5.1
!augustus --species=human --gff3=on genomico.fa > aug_genomico.gff3 2> aug_genomico.log
tipos_gen = summarize_gff("aug_genomico.gff3", "AUGUSTUS - genômico (Parte 2)")
prot_gen = augustus_proteins("aug_genomico.gff3")
print("\nProteína(s) predita(s):", [len(p) for p in prot_gen], "aa")
# Q8: quantos e quais tipos de exons (single, initial, internal, terminal)?
#     tamanho total dos exons? Compare com Q6/Q7.


== AUGUSTUS - genômico (Parte 2) ==
   CDS              12
   gene             1
   start_codon      1
   stop_codon       1
   transcript       1
   -> gene 52004-133460 (-)  81457 pb
   -> transcript 52004-133460 (-)  81457 pb

Proteína(s) predita(s): [424] aa


## 6. ANNEVO — predição *ab initio* por *deep learning*

**ANNEVO** (Chen *et al.*, *Nature Methods*, 2026) é um preditor gênico *ab initio* baseado em um *genomic language model* do tipo *mixture-of-experts*, que modela dependências de longo alcance e relações evolutivas entre espécies diretamente da sequência — sem precisar de evidência externa (RNA-seq, proteínas). Foi avaliado em 566 espécies e tem desempenho comparável a *pipelines* baseados em evidência.

- Repositório: <https://github.com/xjtu-omics/ANNEVO>
- Modelos por linhagem: `Mammalia`, `Aves`, `Actinopteri`, `Insecta`, `Magnoliopsida`, `Fungi` → para *Homo sapiens* usamos **`Mammalia`**.
- Requer **GPU** (~4 GB de VRAM; a T4 do Colab basta). Licença **não comercial** (livre para uso acadêmico).

> ### ⚠️ A ANNEVO precisa de contexto grande
> O modelo prevê em janelas de **~102 kb**. Se a região genômica for muito menor que isso
> (ex.: `FLANK` pequeno), ela roda mas o **GFF sai vazio** — não é erro, é falta de contexto.
> Por isso o `FLANK` padrão da célula 2.2 é **80000**. Se ainda vier vazio, aumente para
> `100000` ou baixe o braço inteiro do cromossomo, e reexecute **2.2 → 3.1 → 5.1 → 6.2 → 6.3**.
> O aviso do DataLoader sobre *"8 worker processes"* é **cosmético** e pode ser ignorado.


In [12]:
# 6.1 — Clona o repositório e instala as dependências que faltam
%cd /content
![ -d ANNEVO ] || git clone --depth 1 https://github.com/xjtu-omics/ANNEVO.git
%cd ANNEVO
import os
mdl = "saved_model/ANNEVO_Mammalia.pt"
sz = os.path.getsize(mdl) / 1e6
print(f"{mdl}: {sz:.1f} MB")
if sz < 1.0:
    print("\n[ATENÇÃO] o arquivo parece ser um ponteiro Git-LFS. Rode:")
    print("   !git lfs install && git lfs pull")
    print("   ou baixe ANNEVO_Mammalia.pt manualmente do repositório/release.")


/content
Cloning into 'ANNEVO'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 56 (delta 3), reused 41 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 318.22 MiB | 16.81 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (43/43), done.
/content/ANNEVO
saved_model/ANNEVO_Mammalia.pt: 58.6 MB


In [13]:
# 6.2 — Roda a ANNEVO sobre a MESMA região genômica da Parte 2 (~5-20 min)
!python annotation.py \
    -g /content/pratica3/genomico.fa \
    -m saved_model/ANNEVO_Mammalia.pt \
    -l Mammalia \
    -o /content/pratica3/annevo_genomico.gff \
    --batch_size 8 -t 2 --overlap_pred --show_log


Prediction window/flank=(102400, 12800), overlap_pred=True
Model loading
Model loading complete
Prediction step_size=51200, overlap_pred=True
---------------------------------------Processing genome information---------------------------------------
Processing genome information took 0.0 seconds
---------------------------------------Prediction on chunk 1---------------------------------------
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0% 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataL

In [14]:
# 6.3 — Resumo da anotação da ANNEVO
%cd /content/pratica3
import os
from collections import Counter

def _tem_features(gff):
    if not os.path.exists(gff) or os.path.getsize(gff) == 0:
        return False
    return any((not l.startswith("#")) and l.strip() for l in open(gff))

if not _tem_features("annevo_genomico.gff"):
    tipos_ann, prot_ann = Counter(), []
    print("A ANNEVO NAO anotou nenhum gene nesta regiao (GFF vazio).\n"
          "Causa mais provavel: a regiao e curta demais para a janela do modelo (~102 kb)\n"
          "e ficou quase toda em 'padding'.\n\n"
          "ACAO: na celula 2.2 aumente FLANK para 80000-100000 e reexecute\n"
          "      2.2 -> 3.1 -> 5.1 -> 6.2 -> 6.3.\n"
          "Confira tambem a celula 7.1: ela mostra se o RefSeq realmente\n"
          "anota um gene na regiao baixada (se nao, o ACCESSION/coordenadas da Q1 estao errados).")
else:
    tipos_ann = summarize_gff("annevo_genomico.gff", "ANNEVO - genômico")
    prot_ann = proteins_from_gff("annevo_genomico.gff", "genomico.fa")
    print("\nProteína(s) predita(s):", [len(p) for p in prot_ann], "aa")
# Q9: quantos exons/CDS? Quanto tempo levou? Como se compara ao AUGUSTUS (Parte 2)?


/content/pratica3
== ANNEVO - genômico ==
   CDS              8
   exon             8
   gene             1
   mRNA             1
   -> gene 52004-133622 (-)  81619 pb
   -> mRNA 52004-133622 (-)  81619 pb

Proteína(s) predita(s): [460] aa


## 7. Comparação das predições

### 7.1 Estrutura de éxons: predições × RefSeq


In [15]:
# 7.1 — Estrutura anotada no RefSeq para a região baixada
gb = SeqIO.read("genomico.gb", "genbank")
print("Features mRNA/CDS anotadas no RefSeq (coordenadas relativas à região baixada):")
for feat in gb.features:
    if feat.type in ("mRNA", "CDS"):
        gene = feat.qualifiers.get("gene", ["?"])[0]
        n = len(feat.location.parts)
        print(f"   {feat.type:5s} {gene:12s} -> {n} segmentos (éxons)  {int(feat.location.start)+1}-{int(feat.location.end)}")
if not any(f.type in ("mRNA", "CDS") for f in gb.features):
    print("   (nenhuma feature mRNA/CDS na região baixada - aumente FLANK na célula 2.2)")


Features mRNA/CDS anotadas no RefSeq (coordenadas relativas à região baixada):
   mRNA  SLC39A8      -> 11 segmentos (éxons)  40001-134213
   CDS   SLC39A8      -> 10 segmentos (éxons)  42382-133622
   mRNA  SLC39A8      -> 11 segmentos (éxons)  46318-134442
   CDS   SLC39A8      -> 10 segmentos (éxons)  46480-133622
   mRNA  SLC39A8      -> 9 segmentos (éxons)  50624-134442
   mRNA  SLC39A8      -> 8 segmentos (éxons)  50624-134151
   mRNA  SLC39A8      -> 10 segmentos (éxons)  50624-133875
   mRNA  SLC39A8      -> 7 segmentos (éxons)  50624-96483
   mRNA  SLC39A8      -> 9 segmentos (éxons)  50626-134442
   mRNA  SLC39A8      -> 8 segmentos (éxons)  50626-113272
   CDS   SLC39A8      -> 8 segmentos (éxons)  52004-133622
   CDS   SLC39A8      -> 8 segmentos (éxons)  52004-133622
   CDS   SLC39A8      -> 8 segmentos (éxons)  52004-133622
   CDS   SLC39A8      -> 8 segmentos (éxons)  52004-113227
   CDS   SLC39A8      -> 8 segmentos (éxons)  52004-113227
   CDS   SLC39A8      -> 5 segme

In [16]:
# 7.2 — Tabela comparativa
import pandas as pd
def _n(t, *keys):
    return sum(t.get(k, 0) for k in keys)
linhas = [
    ("AUGUSTUS  (transcrito)", tipos_tx),
    ("AUGUSTUS  (genômico)",  tipos_gen),
    ("ANNEVO    (genômico)",  tipos_ann),
]
df = pd.DataFrame([
    dict(fonte=nome,
         genes=_n(t, "gene"),
         transcritos=_n(t, "mRNA", "transcript"),
         exons=_n(t, "exon"),
         CDS=_n(t, "CDS"))
    for nome, t in linhas
])
df
# Q11: qual predição chegou mais perto do RefSeq no número de éxons?


,fonte,genes,transcritos,exons,CDS
0,AUGUSTUS (transcrito),1,1,0,1
1,AUGUSTUS (genômico),1,1,0,12
2,ANNEVO (genômico),1,1,8,8


### 7.2 Cada predição × a proteína de **referência** (RefSeq)

Em vez de comparar as predições entre si, alinhamos **cada proteína prevista contra a
proteína RefSeq** do gene (`NP_001128618.1`, SLC39A8/ZIP8, 460 aa). Assim dá para dizer,
para cada caso, **quão perto ficou do "gabarito"**:

- `pident` alto e cobertura ~100 % → a predição reconstruiu a proteína correta;
- `pident` baixo com cobertura alta → sítios de *splice* errados deslocaram o *frame* em
  parte da CDS (é o caso esperado para o AUGUSTUS sobre o genômico);
- cobertura baixa → éxon(s) terminal(is) perdido(s).

A célula abaixo mostra a **tabela-resumo** e o **alinhamento pareado** de cada caso.


In [17]:
# 7.3 — Alinha cada proteína prevista contra a proteína de REFERÊNCIA (RefSeq)
import subprocess
import pandas as pd

REF_PROT = "NP_001128618.1"   # SLC39A8 / ZIP8, isoforma a (460 aa).
                              # Troque se o seu BLAST apontou outra variante.

h = Entrez.efetch(db="protein", id=REF_PROT, rettype="fasta", retmode="text")
ref = SeqIO.read(h, "fasta")
ref_seq = str(ref.seq)
open("ref_prot.fa", "w").write(f">{REF_PROT}\n{ref_seq}\n")
print(f"Referência: {REF_PROT} — {ref.description}")
print(f"           {len(ref_seq)} aa\n")

casos = [
    ("AUGUSTUS_transcrito", prot_tx),
    ("AUGUSTUS_genomico",   prot_gen),
    ("ANNEVO_genomico",     prot_ann),
]

resumo = []
for nome, prots in casos:
    if not prots:
        resumo.append(dict(caso=nome, len_pred=0, pident=None, obs="sem proteína prevista"))
        print(f"===== {nome}: sem proteína prevista =====\n")
        continue
    q = _write_fa([prots[0]], "q_prot.fa", nome)
    campos = "pident length mismatch gapopen gaps qlen slen qstart qend sstart send"
    tab = subprocess.run(
        ["blastp", "-query", q, "-subject", "ref_prot.fa",
         "-outfmt", "6 " + campos],
        capture_output=True, text=True).stdout.strip().splitlines()
    if not tab:
        resumo.append(dict(caso=nome, len_pred=len(prots[0]), pident=None, obs="sem alinhamento"))
    else:
        f = tab[0].split("\t")
        pid, aln_len, mism, gapopen, gaps = float(f[0]), int(f[1]), int(f[2]), int(f[3]), int(f[4])
        resumo.append(dict(
            caso=nome,
            len_pred=len(prots[0]),
            len_ref=len(ref_seq),
            pident=round(pid, 1),
            aln_len=aln_len,
            cobertura_ref_pct=round(100 * aln_len / len(ref_seq), 1),
            mismatches=mism,
            gaps=gaps,
        ))
    # alinhamento pareado legível
    rep = subprocess.run(
        ["blastp", "-query", q, "-subject", "ref_prot.fa", "-outfmt", "0"],
        capture_output=True, text=True).stdout
    trecho = rep.split("Lambda")[0].split("Query=")[-1].strip()
    print(f"===== {nome}  vs  {REF_PROT} =====")
    print(trecho or "(sem alinhamento significativo)")
    print()

pd.DataFrame(resumo)
# Q10: qual(is) predição(ões) reconstruiu(íram) a proteína de referência? Onde a(s) outra(s) errou(aram)?


Referência: NP_001128618.1 — NP_001128618.1 metal cation symporter ZIP8 isoform a precursor [Homo sapiens]
           460 aa

===== AUGUSTUS_transcrito  vs  NP_001128618.1 =====
AUGUSTUS_transcrito1

Length=460
                                                                      Score     E
Sequences producing significant alignments:                          (Bits)  Value

NP_001128618.1                                                        936     0.0  


> NP_001128618.1
Length=460

 Score = 936 bits (2418),  Expect = 0.0, Method: Compositional matrix adjust.
 Identities = 460/460 (100%), Positives = 460/460 (100%), Gaps = 0/460 (0%)

Query  1    MAPGRAVAGLLLLAAAGLGGVAEGPGLAFSEDVLSVFGANLSLSAAQLQHLLEQMGAASR  60
            MAPGRAVAGLLLLAAAGLGGVAEGPGLAFSEDVLSVFGANLSLSAAQLQHLLEQMGAASR
Sbjct  1    MAPGRAVAGLLLLAAAGLGGVAEGPGLAFSEDVLSVFGANLSLSAAQLQHLLEQMGAASR  60

Query  61   VGVPEPGQLHFNQCLTAEEIFSLHGFSNATQITSSKFSVICPAVLQQLNFHPCEDRPKHK  120
            VGVPEPGQLHFNQCLTAEEIFSLHGFSNATQITSS

,caso,len_pred,len_ref,pident,aln_len,cobertura_ref_pct,mismatches,gaps
0,AUGUSTUS_transcrito,460,460,100.0,460,100.0,0,0
1,AUGUSTUS_genomico,424,460,67.0,454,98.7,72,78
2,ANNEVO_genomico,460,460,100.0,460,100.0,0,0


In [18]:
# 7.4 — (opcional) alinhamento GLOBAL do caso mais divergente contra a referência
#        Mostra ONDE a identidade quebra (ex.: bloco traduzido em frame errado).
from Bio.Align import PairwiseAligner, substitution_matrices

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -11
aligner.extend_gap_score = -1

alvo = prot_gen[0] if prot_gen else (prot_ann[0] if prot_ann else None)
if alvo:
    aln = aligner.align(ref_seq, alvo)[0]
    linha_ref, linha_alvo = str(aln[0]), str(aln[1])
    ident = sum(x == y and x != "-" for x, y in zip(linha_ref, linha_alvo))
    cols = len(linha_ref)
    print(f"AUGUSTUS_genomico  vs  {REF_PROT}")
    print(f"Identidade global: {ident}/{cols} = {100 * ident / cols:.1f}%")
    print("(topo = referência RefSeq | base = proteína prevista)\n")
    print(aln)
else:
    print("Nenhuma proteína genômica prevista para alinhar.")


AUGUSTUS_genomico  vs  NP_001128618.1
Identidade global: 304/505 = 60.2%
(topo = referência RefSeq | base = proteína prevista)

target            0 MAPGRAVAGLLLLAAAGLGGVAEGPGLAFSEDVLSVFGANLSLSAAQLQHLLEQMGAASR
                  0 ------------------------------------------------------||||||
query             0 ------------------------------------------------------MGAASR

target           60 VGVPEPGQLHFNQCLTAEEIFSLHGFSNATQITSSKFSVICPAVLQQLNFHPCEDRPKHK
                 60 ||||||||||||------------------|.........|---......-----|....
query             6 VGVPEPGQLHFN------------------QYQKEWYQLLC---IREYRY-----RAVNP

target          120 TRPSHSEVWGYGFLSVTIINLASLL------------GLILTPLIKKSYFPKILTFFVGL
                120 ..|.........||-..|.|....|------------...|....|||......|.....
query            40 SSPGLFLIDLFRFL-LWILNAINFLLNTALAVSQRFWKSVLHLPFKKSFESERRTNLEAI

target          168 AIGTLFSNAIFQLIPEAFGFDPKVDSYVEKAVAVFGGFYLLFFFERMLKMLLKTYGQNGH
                180 ...|...|........|||||||||||||||||||||||

## 8. Discussão — Q10–Q12

- **Q10.** Alinhando **cada proteína prevista contra a referência RefSeq** (célula 7.3):
  quais casos reconstruíram a proteína correta (identidade ~100 %, cobertura ~100 %)?
  No(s) caso(s) com identidade baixa mas cobertura alta, **onde** a sequência diverge
  (célula 7.4) — início/fim, ou um bloco no meio em *frame* trocado? Relacione com o número
  de éxons/CDS previstos.
- **Q11.** Compare o número de éxons das três predições com a anotação RefSeq (célula 7.1)
  e com o número de HSPs do BLAST (Q2). Qual método chegou mais perto? Por quê?
- **Q12.** Qual é o caminho **mais correto** para anotação gênica: rodar o preditor sobre o
  **transcrito** (Parte 1) ou sobre o **genômico** (Parte 2)? Por quê? E qual seria a
  abordagem **ideal** numa anotação real (*ab initio* × baseada em evidência / alinhamento
  *spliced* do transcrito ao genoma)?

**Pontos para amarrar a discussão:**
- O mRNA maduro **já sofreu *splicing*** — rodar o preditor sobre ele só recupera a ORF,
  nunca a estrutura íntron/éxon.
- Sobre o genômico, o preditor precisa **acertar os sítios de *splice*** — um erro de
  junção desloca o *frame* de um trecho da CDS e derruba a identidade contra a referência,
  mesmo com a cobertura quase total.
- ANNEVO (*deep learning*) × AUGUSTUS (HMM): compare identidade contra a referência, número
  de éxons, tempo e dependência de GPU.


## 9. Relatório (preencha e exporte em PDF: *Arquivo → Imprimir → Salvar como PDF*)

| # | Pergunta | Resposta |
|---|---|---|
| Q1 | Cromossomo e coordenadas início/fim do gene | |
| Q2 | Nº de éxons pelo alinhamento BLASTN (HSPs) | |
| Q3 | Fita do gene (Plus/Plus ou Plus/Minus) | |
| Q4 | Gene e função | |
| Q5 | Posição do sinal AATAAA e tamanho da 3′UTR | |
| Q6 | Importância de flancos | |
| Q7 | AUGUSTUS/transcrito: nº e tipos de éxons; há íntrons? | |
| Q8 | AUGUSTUS/transcrito: tamanho total dos éxons | |
| Q9 | AUGUSTUS/genômico: nº e tipos de éxons; tamanho total | |
| Q10 | ANNEVO/genômico: nº de éxons/CDS; tempo; vs AUGUSTUS | |
| Q11 | As proteínas são iguais? O que aconteceu? | |
| Q12 | Nº de éxons das predições vs RefSeq — qual acertou mais? | |
| Q13 | Melhor caminho de anotação + abordagem ideal | |

---

### Referências
- Chen *et al.* **Highly accurate *ab initio* gene annotation with ANNEVO.** *Nature Methods* (2026). <https://www.nature.com/articles/s41592-026-03036-7>
- ANNEVO — código e modelos: <https://github.com/xjtu-omics/ANNEVO>
- Stanke *et al.* **AUGUSTUS.** <https://bioinf.uni-greifswald.de/augustus/>
- NCBI BLAST: <https://blast.ncbi.nlm.nih.gov/> · UCSC BLAT: <https://genome.ucsc.edu/cgi-bin/hgBlat>

---

### Gabarito (para o instrutor)

| Item | Valor de referência |
|---|---|
| Organismo | *Homo sapiens* |
| Transcrito | **NM_001135146.2** (variante 2), 3187 nt, cauda poli-A |
| Gene | **SLC39A8** / ZIP8 (Gene ID 64116), *solute carrier family 39 member 8* |
| Função | transportador de metais de transição (Zn²⁺, Mn²⁺, também Cd²⁺); membrana plasmática/mitocôndria; papel na inflamação |
| Localização | cromossomo **4q24**, **NC_000004.12** (GRCh38.p14) |
| Coordenadas | **102 251 041 – 102 345 482** (~94,4 kb) |
| Fita | **minus** (BLAST: Strand = Plus/Minus) |
| Éxons | ~10–11 por transcrito (17 no locus somando variantes); ~10 éxons codificantes |
| Montagem | fixar **GRCh38.p14 / NC_000004.12** no enunciado — as coordenadas da Q1 dependem dela |

> **Nota didática:** a Parte 1 (AUGUSTUS sobre o mRNA) tende a devolver **1 éxon único, sem íntrons** — o transcrito já sofreu *splicing*. A Parte 2 e a ANNEVO (sobre o genômico) devem recuperar a estrutura multi-éxon. Diferenças de proteína no “Blast2seq” geralmente vêm de *start*/*stop* ou de um éxon terminal previsto de forma diferente.
